# Lesson 07 - ప్లానింగ్ డిజైన్ ప్యాటర్న్

ఈ నోట్‌బుక్ మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్ ఉపయోగించి AI ఏజెంట్ల కోసం **ప్లానింగ్ డిజైన్ ప్యాటర్న్** ను ప్రదర్శిస్తుంది.  
మీరు సవాలైన ప్రయాణ అభ్యర్థనలను నిర్మిత సబ్‌టాస్క్‌లుగా విభజించడం, వాటికి ప్రత్యేక ఏజెంట్లను కేటాయించడం,  
మరియు ఫలితంగా ఏర్పడిన ప్లాన్‌ను అమలు చేయడం - ఇవన్నీ Pydantic మోడల్స్ ద్వారా శక్తివంతమైన నిర్మిత అవుట్పుట్ ఉపయోగించి ఎలా చేయాలో నేర్చుకుంటారు.


## సెటప్


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Task Decomposition

Task decomposition అనేది ప్లానింగ్ డిజైన్ ప్యాటర్న్ యొక్క హృదయం. ఒకే ఏజెంట్‌ను క్లిష్టమైన అభ్యర్థనను మొత్తం ఈండ్-టు-ఈండ్ హ్యాండిల్ చేయమని అడుగుతున్నట్లు కాకుండా, సమస్యను చిన్న, బాగా నిర్వచించబడిన **subtasks** గా విభజిస్తాము. ప్రతి సబ్‌టాస్క్‌ను ప్రత్యేక నிபుణ agent (ఉదా: ఫ్లైట్స్, హోటల్స్, యాక్టివిటీస్) పంపిణీ చేస్తారు, స్పష్టమైన ప్రాధాన్యతలు మరియు ఆధారిత ఆర్డరింగ్ తో.

ఈ పద్ధతి కొన్ని ప్రయోజనాలను అందిస్తుంది:  
- **స్పష్టత**: ప్రతి సబ్‌టాస్క్‌కు ఒకకాని బాధ్యత ఉంటుంది.  
- **పారాలలిజం**: స్వతంత్ర సబ్‌టాస్క్‌లు సమాంతరంగా నడవచ్చు.  
- **భరోసా**: విఫలతలు వ్యక్తిగత సబ్‌టాస్క్‌లకు పరిమితం అవుతాయి.  
- **బడ్జెట్ ట్రాకింగ్**: ఖర్చులు ప్రతి సబ్‌టాస్క్‌కి అంచనా వేయబడతాయి మరియు సారాంశం చేయబడతాయి.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## రాచనా అవుట్పుట్‌తో ప్లానింగ్ ఏజెంట్ సృష్టించడం

ప్లానింగ్ ఏజెంట్ ఒక **ఫ్రంట్ డెస్క్ కోఆర్డినేటర్** గా పనిచేస్తుంది. ఒక హై-లెవల్ ట్రావెల్ అభ్యర్థన ఇచ్చినపుడు ఇది ఒక రాచనా `TravelPlan` ని ఉత్పత్తి చేస్తుంది — అభ్యర్థనను ఉప కార్యాలయాలలో విభజించి, ప్రాధాన్యతలను నిర్ణయించి, అధిని గుర్తించి, కిబట్టి ఒక కాన్సెర్జ్ లేదా కార్యాచరణ స్థాయి పని నిర్వహించేందుకు సహాయపడుతుంది.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## నైపుణ్యపూరిత సాధనాలతో ఒక ప్రణాళికను అమలు చేయడం

ఫ్రంట్ డెస్క్ ఏజెంట్ ఒక నిర్మిత ప్రణాళికను తయారుచేసిన తర్వాత, **కాన్సియర్ ఏజెంట్** దాన్ని అమలు చేస్తుంది.
ప్రతి నైపుణ్య సాధనం ఒక ఉపకర్తవ్యపు వర్గాన్ని (ఫ్లైట్స్, హోటల్స్, కార్యకలాపాలు) నిర్వహిస్తుంది. కాన్సియర్ ప్రణాళికలోని ఉపకర్తవ్యాలను ఆధారపడే క్రమంలో తిరిగి చూసి ప్రతి ఒక్కటిని సరైన సాధనానికి పంపిస్తాడు.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## సారాంశం

ఈ పాఠంలో మీరు AI ఏజెంట్ల కోసం **ప్లానింగ్ డిజైన్ ప్యాటర్న్** నేర్చుకున్నారు:

1. **టాస్క్ డీకంపోజిషన్** — ఒక ఫ్రంట్ డెస్క్ ప్లానింగ్ ఏజెంట్ సంక్లిష్ట ప్రయాణ అభ్యర్థనను పధ్యమైన ఉపటాస్క్‌లుగా విభజించి, ప్రతి టాస్క్‌ను ప్రాధాన్యతలు మరియు ఆధారితతలతో ప్రత్యేక నిపుణ ఏజెంట్‌కి అప్పగిస్తుంది.
2. **సంఘటిత అవుట్‌పుట్** — `response_format`ను ఉపయోగించి ఏజెంట్ స్వేచ్ఛా రూపమైన టెక్స్ట్ కాకుండా ధృవీకరించిన `TravelPlan` వస్తువును పంపించి, తదుపరి ప్రాసెసింగ్‌ను నమ్మకంగా చేస్తుంది.
3. **ప్లాన్ అమలు** — ఒక కాంసియర్జ్ ఏజెంట్ ఉపటాస్క్‌ల ద్వారా నిపుణ సాధనాలు (`book_flight`, `reserve_hotel`, `book_activity`) ఉపయోగించి ప్లాన్‌ను అమలు చేసి ఫలితాలను నివేదిస్తుంది.

ఈ ప్యాటర్న్ *ఏమి చేయాలో* (ప్లానింగ్) మరియు *ఎలా చేయాలో* (అమలు) విడగొట్టి, ఏజెంట్లను మరింత మాడ్యులర్, పరీక్షించటం సులభం మరియు విస్తరించటం సులభం చేస్తుంది.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అस्वీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నించినప్పటికీ, ఆటోమేటిక్ అనువాదాలలో తప్పులు లేదా అకృతతలు ఉండవచ్చు. ఒరిజినల్ పత్రాన్ని దాని స్థానిక భాషలోనే అధికారిక మూలంగా పరిగణించాలి. ముఖ్యమైన సమాచారానికి, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫార్సు చేయబడుతుంది. ఈ అనువాదం వాడకంతో ఉత్పన్నమయ్యే ఏవైనా అవగాహనా లోపాలు లేదా తప్పుదరువుల కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
